In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
model_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/path_to_your_best.pt'
test_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/path_to_your_test_true.txt'


In [16]:
torch.save(model.state_dict(), 'best.pt')


In [27]:
# 打印文件内容
def print_file_contents(file_path):
    try:
        with open(file_path, encoding='utf-8', errors='ignore') as f:
            print("文件内容:")
            for line in f.readlines():
                print(line.strip())
    except Exception as e:
        print(f"Error reading file: {e}")

# 打印 test_true.txt 文件内容
test_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/test_true.txt'  # 确保路径正确
print_file_contents(test_file_path)


文件内容:
.\ʳ\\ޱ1.jpeg 11
.\ʳ\\ޱ2.jpg 11
.\ʳ\\ޱ3.jpg 11
.\ʳ\\ޱ4.jpg 11
.\ʳ\\ޱ5.jpg 4
.\ʳ\\ޱ6.jpg 4
.\ʳ\\ޱ7.jpg 4
.\ʳ\\ޱ8.jpg 4
.\ʳ\\ޱ9.jpg 12
.\ʳ\\ޱ10.jpg 12
.\ʳ\\ޱ11.jpg 12
.\ʳ\\ޱ12.jpg 12
.\ʳ\\ޱ13.jpg 3
.\ʳ\\ޱ14.jpg 3
.\ʳ\\ޱ15.jpg 3
.\ʳ\\ޱ16.jpg 3
.\ʳ\\ޱ17.jpg 3
.\ʳ\\ޱ18.jpg 2
.\ʳ\\ޱ19.jpg 2
.\ʳ\\ޱ20.jpg 2
.\ʳ\\ޱ21.jpg 2
.\ʳ\\ޱ22.jpg 9
.\ʳ\\ޱ23.jpg 9
.\ʳ\\ޱ24.jpg 9
.\ʳ\\ޱ25.jpg 9
.\ʳ\\ޱ26.jpg 14
.\ʳ\\ޱ27.jpg 14
.\ʳ\\ޱ28.jpg 14
.\ʳ\\ޱ29.jpg 14
.\ʳ\\ޱ30.jpg 15
.\ʳ\\ޱ31.jpg 15
.\ʳ\\ޱ32.jpg 15
.\ʳ\\ޱ33.jpg 15
.\ʳ\\ޱ34.jpg 17
.\ʳ\\ޱ35.jpg 17
.\ʳ\\ޱ36.jpg 17
.\ʳ\\ޱ37.jpg 17
.\ʳ\\ޱ38.jpg 7
.\ʳ\\ޱ39.jpg 7
.\ʳ\\ޱ40.jpg 7
.\ʳ\\ޱ41.jpg 7
.\ʳ\\ޱ42.jpg 0
.\ʳ\\ޱ43.jpg 0
.\ʳ\\ޱ44.jpg 0
.\ʳ\\ޱ45.jpg 0


In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from torchvision import transforms

# 挂载Google云盘
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 更新文件路径
train_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/train.txt'
test_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/test_true.txt'
model_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/best.pt'

# 定义数据转换 (假设你已经定义了 data_transforms)
# 例如:
# data_transforms = {
#     'train': transforms.Compose([
#         transforms.RandomResizedCrop(224),
#         transforms.RandomHorizontalFlip(),
#         transforms.ToTensor(),
#         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
#     ]),
#     'valid': transforms.Compose([
#         transforms.Resize(256),
#         transforms.CenterCrop(224),
#         transforms.ToTensor(),
#         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
#     ]),
# }


# 自定义数据集类
class food_dataset(Dataset):
    def __init__(self, file_path, transform=None):
        self.transform = transform  # 数据转换操作
        self.imgs = []  # 存储图片路径
        self.labels = []  # 存储图片标签

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip().split('\t')  # 使用制表符分割
                self.imgs.append(line[0])
                self.labels.append(int(line[1]))  # 将标签转换为整数


    def __len__(self):
        return len(self.imgs)  # 返回数据集大小


    def __getitem__(self, idx):
        image_path = self.imgs[idx]
        print(f"Loading image: {image_path}")  # 打印文件路径

        # 检查文件是否存在
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"File does not exist: {image_path}")

        image = Image.open(image_path)
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        label = torch.from_numpy(np.array(label, dtype=np.int64))
        return image, label


# 准备训练和测试数据
training_data = food_dataset(file_path=train_file_path, transform=data_transforms['train'])
test_data = food_dataset(file_path=test_file_path, transform=data_transforms['valid'])

# 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# 定义CNN模型
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2),
            nn.ReLU(),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self

In [30]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from torchvision import transforms

# 挂载Google云盘
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 更新文件路径
train_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/train.txt'
test_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/test_true.txt'
model_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/best.pt'

# 定义数据转换
# ... (data_transforms 保持不变) ...

# 自定义数据集类
class food_dataset(Dataset):
    def __init__(self, file_path, transform=None):
        # ... (__init__ 的其他部分保持不变) ...

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        image_path = self.imgs[idx]
        #  修改此处，使用完整路径，而不是拼接路径
        # image_path = os.path.join('/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train', os.path.basename(image_path))
        print(f"Loading image: {image_path}")  # 打印文件路径

        # 检查文件是否存在
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"File does not exist: {image_path}")

        image = Image.open(image_path)
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        label = torch.from_numpy(np.array(label, dtype=np.int64))
        return image, label

# ... (其余代码保持不变) ...

# 准备训练和测试数据
training_data = food_dataset(file_path=train_file_path, transform=data_transforms['train'])
test_data = food_dataset(file_path=test_file_path, transform=data_transforms['valid'])

# 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# 定义CNN模型
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2),
            nn.ReLU(),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU()
        )
        self.out = nn.Linear(64 * 64 * 64, 20)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        output = self.out(x)
        return output

# 创建模型实例并加载整个模型对象
model = torch.load(model_path, map_location=device)
model.to(device)
model.eval()

# 加载测试数据
test_data = food_dataset(file_path=test_file_path, transform=data_transforms['valid'])
test_dataloader = DataLoader(test_data, batch_size=1, shuffle=True)

# 存储预测结果和真实标签
result = []
labels = []

# 定义测试函数
def test_true(dataloader, model):
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            result.append(pred.argmax(1).item())
            labels.append(y.item())

# 运行测试
test_true(test_dataloader, model)
print('真实预测值：\t', result)
print('真实值：\t', labels)


IndentationError: expected an indented block after function definition on line 23 (<ipython-input-30-46e3c206de73>, line 26)

In [26]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from torchvision import transforms

# 挂载Google云盘
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 更新文件路径
train_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/train.txt'
test_file_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/test_true.txt'
model_path = '/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/best.pt'

# 定义数据转换
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize([300, 300]),
        transforms.RandomRotation(45),
        transforms.CenterCrop(256),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize([256, 256]),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456], [0.229, 0.224, 0.225])
    ]),
}

# 自定义数据集类
class food_dataset(Dataset):
    def __init__(self, file_path, transform=None):
        self.file_path = file_path
        self.imgs = []
        self.labels = []
        self.transform = transform
        with open(self.file_path, encoding='utf-8', errors='ignore') as f:
            samples = [x.strip().split(' ') for x in f.readlines()]
            for sample in samples:
                img_path = ' '.join(sample[:-1])  # 合并路径部分
                img_path = os.path.normpath(img_path)  # 标准化路径
                img_path = img_path.replace("\\", "/")  # 替换反斜杠为正斜杠
                self.imgs.append(img_path)
                label = sample[-1]  # 最后的部分作为标签
                self.labels.append(label)

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        image_path = self.imgs[idx]
        print(f"Loading image: {image_path}")  # 打印文件路径
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"File does not exist: {image_path}")
        image = Image.open(image_path)
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        label = torch.from_numpy(np.array(label, dtype=np.int64))
        return image, label

# 准备训练和测试数据
training_data = food_dataset(file_path=train_file_path, transform=data_transforms['train'])
test_data = food_dataset(file_path=test_file_path, transform=data_transforms['valid'])

# 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# 定义CNN模型
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2),
            nn.ReLU(),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU()
        )
        self.out = nn.Linear(64 * 64 * 64, 20)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        output = self.out(x)
        return output

# 创建模型实例并加载整个模型对象
model = torch.load(model_path, map_location=device)
model.to(device)
model.eval()

# 加载测试数据
test_data = food_dataset(file_path=test_file_path, transform=data_transforms['valid'])
test_dataloader = DataLoader(test_data, batch_size=1, shuffle=True)

# 存储预测结果和真实标签
result = []
labels = []

# 定义测试函数
def test_true(dataloader, model):
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            result.append(pred.argmax(1).item())
            labels.append(y.item())

# 运行测试
test_true(test_dataloader, model)
print('真实预测值：\t', result)
print('真实值：\t', labels)


Mounted at /content/drive
Using cpu device
Loading image: ./ʳ//ޱ25.jpg


<ipython-input-26-d99e7c694e67>:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path, map_location=device)


FileNotFoundError: File does not exist: ./ʳ//ޱ25.jpg